In [21]:
import pandas as pd
import numpy as np

from pathlib import Path

In [24]:
GEPA_RUN  = "B8_H200mini_V1"
GEPA_PATH = Path(f"GEPA_runs/{GEPA_RUN}")

In [25]:
# %run ../gs_slimming.py
# print("="*120)
# print("gs_slimming done.")
# print("="*120)
%run GEPA_runs/flattening.py -p "{GEPA_PATH}"
print("="*120)
print("flattening done.")
print("="*120)


  Flattening 5 JSONs => GEPA_runs/B8_H200mini_V1/0.csv
  Reports    : 5
  Zeilen     : 115
  Fehler     : 0
  Gespeichert: GEPA_runs/B8_H200mini_V1/0.csv

  Flattening 5 JSONs => GEPA_runs/B8_H200mini_V1/1.csv
  Reports    : 5
  Zeilen     : 115
  Fehler     : 0
  Gespeichert: GEPA_runs/B8_H200mini_V1/1.csv

  Flattening 5 JSONs => GEPA_runs/B8_H200mini_V1/10.csv
  Reports    : 5
  Zeilen     : 91
  Fehler     : 0
  Gespeichert: GEPA_runs/B8_H200mini_V1/10.csv

  Flattening 5 JSONs => GEPA_runs/B8_H200mini_V1/2.csv
  Reports    : 5
  Zeilen     : 59
  Fehler     : 0
  Gespeichert: GEPA_runs/B8_H200mini_V1/2.csv

  Flattening 5 JSONs => GEPA_runs/B8_H200mini_V1/3.csv
  Reports    : 5
  Zeilen     : 98
  Fehler     : 0
  Gespeichert: GEPA_runs/B8_H200mini_V1/3.csv

  Flattening 5 JSONs => GEPA_runs/B8_H200mini_V1/4.csv
  Reports    : 5
  Zeilen     : 47
  Fehler     : 0
  Gespeichert: GEPA_runs/B8_H200mini_V1/4.csv

  Flattening 5 JSONs => GEPA_runs/B8_H200mini_V1/5.csv
  Reports    : 5

## Import slimmed down Gold-Standard

In [26]:
gs = pd.read_json("../gs_slim.json")
gs["year"] = gs["year"].astype(str) #Correction needed for e.g. fiscal years "FY 2021/2022"

## Import all 3x2=6 Extractions

In [5]:
CSV_LIST = sorted(
    [p for p in GEPA_PATH.glob("*.csv") if p.stem.isdigit()],
    key=lambda p: int(p.stem)
)

oa = {i: pd.read_csv(CSV_LIST[i]) for i in range(len(CSV_LIST))}

df_to_merge = [(oa[i], f"_{i}") for i in range(len(CSV_LIST))]
## this makes the same as
# df_to_merge = [
#     (think1, "_t1"),
#     (think2, "_t2"),
#     (moe1, "_m1"),
#     (moe2, "_m2"),
#     (instr1, "_i1"),
#     (instr2, "_i2")
# ]
## but for the many GEPA iterations (up to 30 at the moment)

In [6]:
extracted_report_names = set()
for df, _ in df_to_merge:
    extracted_report_names.update(df["report_name"].unique())

gs = gs[gs["report_name"].isin(extracted_report_names)].reset_index(drop=True)

print(f"{len(extracted_report_names)} extracted reports:")
print(sorted(extracted_report_names))

5 extracted reports:
['Allianz_2022_report', 'addtech_2022_report', 'fujifilm_2022_report', 'jetblue airways corp_2019_report', 'sato oyj_2022_report']


In [7]:
# Year normalization RegEx-Script

import re

def normalize_year(raw: str, years_present: set[int] | None = None) -> tuple[int | None, str]:
    """Map a raw fiscal-year label to a calendar year. Returns (year, rule_applied)."""
    label = raw.strip().upper().removeprefix("FY").strip()

    if re.fullmatch(r"\d{4}", label):
        return int(label), "plain"
    if re.fullmatch(r"\d{2}", label):
        return 2000 + int(label), "fy_2digit"

    m = re.fullmatch(r"(\d{4})/(\d{1,4})", label)
    if m:
        left, right = m.groups()
        if len(right) == 4:
            return int(right), "range_later_year"  # 2021/2022 -> 2022
        candidates = {int(left), int(left) + 1}
        if years_present:
            hit = candidates & years_present
            if len(hit) == 1:
                return hit.pop(), "fy_end_month_context"
        return int(left), "fy_end_month_fallback"

    return None, "unparseable"

In [8]:
# Get years from extractions 
years_in_extractions = set()
for df, _ in df_to_merge:
    years_in_extractions.update(df["year"].unique().tolist())

# Create ynrom DataFrames to preserve the original ones
oa_copies = {i: oa[i].copy() for i in range(len(CSV_LIST))}

df_to_merge_ynorm = [(oa_copies[i], f"_{i}") for i in range(len(CSV_LIST))]

# Now normalization via the RegEx Script from above
for df, _ in df_to_merge_ynorm:
    df["year"], df["year_rule"] = zip(*df["year"].apply(
        normalize_year,
        years_present=years_in_extractions
    ))

## Matching Extractions to Gold-Standard Scheme

### Before ynrom

In [9]:
years = set()
year_report = []

for df, _ in df_to_merge:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements
    
print(sorted(years))

['2015', '2016', '2017', '2018', '2019', '2019/2020', '2020', '2020/2021', '2021', '2021/2022', '2022']


### After ynrom

In [10]:
years = set()
year_report = []

for df, _ in df_to_merge_ynorm:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements
    
print(sorted(years))

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


## Mapping special occurences

In [11]:
scope_mapping = {
    "scope_1": "1",
    "scope_2_location_based": "2lb",
    "scope_2_market_based" : "2mb",
    "scope_3" : "3"
}

# Prints out only if sth. goes wrong
for df, sf in df_to_merge:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. ursprüngliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")



Total NaN nach Konversion: 0


In [12]:
df_to_merge_ynorm[0]

(                          report_name                   scope  year    value  \
 0                 Allianz_2022_report                 scope_1  2022    30953   
 1                 Allianz_2022_report                 scope_1  2021    28699   
 2                 Allianz_2022_report                 scope_1  2020    28714   
 3                 Allianz_2022_report    scope_2_market_based  2022    30490   
 4                 Allianz_2022_report    scope_2_market_based  2021    54689   
 ..                                ...                     ...   ...      ...   
 110  jetblue airways corp_2019_report                 scope_3  2019  1772985   
 111              sato oyj_2022_report                 scope_1  2022       20   
 112              sato oyj_2022_report    scope_2_market_based  2022    37920   
 113              sato oyj_2022_report  scope_2_location_based  2022    40393   
 114              sato oyj_2022_report                 scope_3  2022       49   
 
         unit             

In [13]:

# Prints out only if sth. goes wrong
for df, sf in df_to_merge_ynorm:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. ursprüngliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")



Total NaN nach Konversion: 0


Checking if all `report_names` are the same

And checking if all `report_names` from the extractions are exactly matched in the GoldStandard

## Merging Extraction Values and Gold-Standard

In [14]:
merge_on = ["report_name", "scope", "year"]
agg_cols = ["value", "unit", "label"]

In [15]:
merged = gs.copy() # Setting a starting point for the loop. Everything has to be mapped to the Gold-Standard

for df, sf in df_to_merge:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged = pd.merge(merged, agg, on=merge_on, how="left")

merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Data columns (total 44 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      226 non-null    str    
 1   year             226 non-null    str    
 2   scope            226 non-null    str    
 3   page             115 non-null    str    
 4   value            115 non-null    float64
 5   unit             115 non-null    str    
 6   unit_normalized  115 non-null    str    
 7   label            115 non-null    str    
 8   status           226 non-null    str    
 9   scopes_present   226 non-null    object 
 10  years_present    226 non-null    object 
 11  value_0          78 non-null     object 
 12  unit_0           78 non-null     object 
 13  label_0          78 non-null     object 
 14  value_1          78 non-null     object 
 15  unit_1           78 non-null     object 
 16  label_1          78 non-null     object 
 17  value_2          73 non-nul

In [16]:
merged["report_name"].unique()

<StringArray>
[             'addtech_2022_report',              'Allianz_2022_report',
             'fujifilm_2022_report', 'jetblue airways corp_2019_report',
             'sato oyj_2022_report']
Length: 5, dtype: str

In [17]:
merged_ynorm = gs.copy()
merged_ynorm["year"] = merged_ynorm["year"].astype(int)

for df, sf in df_to_merge_ynorm:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged_ynorm = pd.merge(merged_ynorm, agg, on=merge_on, how="left")

merged_ynorm.info()

<class 'pandas.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Data columns (total 44 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      226 non-null    str    
 1   year             226 non-null    int64  
 2   scope            226 non-null    str    
 3   page             115 non-null    str    
 4   value            115 non-null    float64
 5   unit             115 non-null    str    
 6   unit_normalized  115 non-null    str    
 7   label            115 non-null    str    
 8   status           226 non-null    str    
 9   scopes_present   226 non-null    object 
 10  years_present    226 non-null    object 
 11  value_0          89 non-null     object 
 12  unit_0           89 non-null     object 
 13  label_0          89 non-null     object 
 14  value_1          89 non-null     object 
 15  unit_1           89 non-null     object 
 16  label_1          89 non-null     object 
 17  value_2          85 non-nul

#### Rows where no value is present must be from Type "NaN" for better analysis; not "None" because the object is missing.

In [18]:
pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

for col in pipeline_cols:
    merged[col] = merged[col].apply(
    lambda x: np.nan if x is None else x
)
    
for col in pipeline_cols:
    merged_ynorm[col] = merged_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

### Rearranging Columns

In [19]:

gs_cols = [
    'report_name', 'status', 'scopes_present', 'years_present', # Per-Report granularity
    'year', 'scope',                                            # Per category
    'page', 'value', 'unit',                                    # About the value as-is in the report
    'unit_normalized', 'label',                                 # Additional information
]

pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

merged = merged[gs_cols + pipeline_cols]
merged_ynorm = merged_ynorm[gs_cols + pipeline_cols]

merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Data columns (total 44 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      226 non-null    str    
 1   status           226 non-null    str    
 2   scopes_present   226 non-null    object 
 3   years_present    226 non-null    object 
 4   year             226 non-null    str    
 5   scope            226 non-null    str    
 6   page             115 non-null    str    
 7   value            115 non-null    float64
 8   unit             115 non-null    str    
 9   unit_normalized  115 non-null    str    
 10  label            115 non-null    str    
 11  value_0          78 non-null     object 
 12  value_1          78 non-null     object 
 13  value_2          73 non-null     object 
 14  value_3          50 non-null     object 
 15  value_4          49 non-null     object 
 16  value_5          49 non-null     object 
 17  value_6          50 non-nul

## Saving created DataFrame

In [20]:
merged.to_json(f"{GEPA_RUN}.json", index=False, orient="records", indent=4)
merged_ynorm.to_json(f"{GEPA_RUN}_ynorm.json", index=False, orient="records", indent=4)

## To later read back the files:
# pd.read_csv("gs_extractions_raw.csv")   # Lists as Strings. Needs ast.literal_eval
# pd.read_json("gs_extractions_raw.json", orient="records")  # Lists instant usable
